# **County Propensity Score Calculation**

This notebook computes county-level solar siting propensity scores using the reported coefficients from the Capacity Intensity regression results.

It produces two versions of the score:

- a social-only propensity score using socio-political variables
- a full propensity score using both techno-economic and socio-political variables

The notebook then normalizes these scores and exports county-level CSV files for later backtesting and model input.

In [50]:
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# If this notebook is in a separate repo next to Solar-NIMBY:
basepath = "../Solar-NIMBY/data/"

# Main county-level inputs from Solar-NIMBY repo
tech = pd.read_csv(basepath + "suitability_scores/suitability_scores_county.csv")
social = pd.read_csv(basepath + "county_clean/social_factors_merged.csv")

print("Loaded tech shape:", tech.shape)
print("Loaded social shape:", social.shape)

display(tech.head())
display(social.head())

Loaded tech shape: (3233, 9)
Loaded social shape: (3125, 66)


,GHI,Protected_Land,Habitat,Slope,Population_Density,Distance_to_Substation,Land_Cover,County Name,State
0,15.0,91.547835,33.748285,91.029530,98.634118,53.241671,67.860969,Ballard,Kentucky
1,15.0,99.583685,82.840678,71.723980,96.682895,60.582990,85.691192,Bourbon,Kentucky
2,15.0,99.896136,39.236428,42.759798,98.239510,50.000000,61.839634,Butler,Kentucky
3,15.0,97.733883,29.826520,11.827322,93.352800,52.380053,56.737948,Estill,Kentucky
4,15.0,98.000821,49.732293,32.705892,99.466845,50.000000,71.555286,Fleming,Kentucky


,GEOID,State,County Name,area km2,area mi2,FIPS State,FIPS County,Wind Capacity Intensity (MW / 1000 sq mile),Wind Project Intensity (Projects / 1000 sq mile),Wind Avg Capacity Intensity (MW / 1000 sq mile),GDP_2017,GDP_2018,GDP_2019,GDP_2020,GDP_2021,GDP_2022,Solar MW 1000 sq mile all,Solar Projects 1000 sq mile all,Solar MW Avg 1000 sq mile all,Solar MW 1000 sq mile small,Solar Projects 1000 sq mile small,Solar MW Avg 1000 sq mile small,Solar MW 1000 sq mile medium,Solar Projects 1000 sq mile medium,Solar MW Avg 1000 sq mile medium,Solar MW 1000 sq mile large,Solar Projects 1000 sq mile large,Solar MW Avg 1000 sq mile large,No. of Private Schools,Median Income,Total Unemployment,Unemployment Rate,Hispanic/Latino,White,Black/African American,American Indian/Alaska Native,Asian,Native Hawaiian/Other Pacific Islander,Others,Number of Existing Installs,Total Installed Capacity (kW),Median Installed Capacity (kW),Total Installed Capacity (kW/ 1000 sq mile),Median Installed Capacity (kW / sq mile),Number of Existing Installs / sq mile,democrat_percentage_vote,republican_percentage_vote,green_percentage_vote,libertarian_percentage_vote,other_percentage_vote,18-24 Less than high school graduate,18-24 High school graduate,18-24 Some college or associate's degree,18-24 Bachelor's degree or higher,25+ Less than 9th grade,"25+ 9th to 12th grade, no diploma",25+ High school graduate,"25+ Some college, no degree",25+ Associate's degree,25+ Bachelor's degree,25+ Graduate or professional degree,25+ High school graduate or higher,25+ Bachelor's degree or higher,Electric Commercial Rate,Electric Industrial Rate,Electric Residential Rate
0,1001.0,Alabama,Autauga,1565.322757,604.374247,1.0,1.0,NaN,NaN,NaN,29.49,29.91,28.96,28.82,28.91,32.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,62660,46577.0,2.8,0.036000,0.707117,0.193045,0.003129,0.014846,0.000374,0.043823,12.0,227901.00,12.75,377085.89,0.02,0.02,0.270184,0.714368,NaN,NaN,0.015448,9.9,33.4,45.4,11.3,2.0,8.4,32.8,19.6,9.1,16.4,11.7,89.6,28.1,0.121895,0.063652,0.135057
1,1003.0,Alabama,Baldwin,4352.548564,1680.527706,1.0,3.0,NaN,NaN,NaN,29.96,31.48,33.07,32.88,35.46,36.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,64346,183804.0,3.7,0.054736,0.804666,0.077669,0.005570,0.008754,0.000526,0.046284,35.0,1123484.25,14.50,668530.63,0.01,0.02,0.224090,0.761714,NaN,NaN,0.014196,14.4,39.1,38.4,8.1,2.1,6.9,27.4,21.7,9.5,20.6,11.8,91.0,32.5,0.121895,0.063652,0.135057
2,1005.0,Alabama,Barbour,2342.545642,904.461557,1.0,5.0,NaN,NaN,NaN,30.83,31.33,30.87,29.61,30.27,30.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,36422,20639.0,8.6,0.059866,0.439519,0.469809,0.002299,0.004084,0.000000,0.023629,NaN,NaN,NaN,NaN,NaN,NaN,0.457882,0.534512,NaN,NaN,0.007606,23.2,40.3,33.1,3.5,7.4,16.9,36.7,20.5,7.3,6.7,4.4,75.7,11.2,0.121895,0.063652,0.135057
3,1007.0,Alabama,Bibb,1622.295670,626.371603,1.0,7.0,NaN,NaN,NaN,18.48,18.05,20.06,20.91,20.82,20.38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54277,18333.0,9.7,0.033194,0.737541,0.196923,0.001749,0.001166,0.000404,0.027632,0.0,305.00,18.75,486.93,0.03,0.00,0.206983,0.784263,NaN,NaN,0.008755,14.1,64.3,20.1,1.6,6.2,13.3,43.9,18.0,6.7,7.9,4.0,80.5,11.9,0.121895,0.063652,0.135057
4,1009.0,Alabama,Blount,1685.098070,650.619735,1.0,9.0,NaN,NaN,NaN,16.60,17.59,17.05,15.24,17.23,17.57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,52830,46871.0,6.0,0.097592,0.841546,0.013968,0.003179,0.002942,0.000186,0.039284,NaN,NaN,NaN,NaN,NaN,NaN,0.095694,0.895716,NaN,NaN,0.008591,16.5,43.6,36.3,3.6,6.8,9.6,35.1,21.5,12.1,9.3,5.6,83.6,14.9,0.121895,0.063652,0.135057


In [51]:
# ---------------------------------------------------------
# COEFFICIENTS FROM TABLE 3
# Capacity Intensity regression
# ---------------------------------------------------------

INTERCEPT = -52.397

# -----------------------------
# Techno-economic factors
# -----------------------------
TECH_COEFS = {
    "GHI": 0.902,
    "Unprotected_Land": 0.287,
    "Slope": 0.226,
    "Population_Sparsity": 0.191,
}

# -----------------------------
# Socio-political factors
# -----------------------------
SOCIAL_COEFS = {
    "Black": 0.877,
    "Asian": 0.449,
    "Income": 0.459,
    "Grad_Education": -0.495,
    "GDP_per_Capita": 0.810,
}

print("Techno-economic coefficients:")
for k, v in TECH_COEFS.items():
    print(f"  {k}: {v}")

print("\nSocio-political coefficients:")
for k, v in SOCIAL_COEFS.items():
    print(f"  {k}: {v}")

Techno-economic coefficients:
  GHI: 0.902
  Unprotected_Land: 0.287
  Slope: 0.226
  Population_Sparsity: 0.191

Socio-political coefficients:
  Black: 0.877
  Asian: 0.449
  Income: 0.459
  Grad_Education: -0.495
  GDP_per_Capita: 0.81


In [52]:
def clean_geoid_5(series):
    return (
        series.astype(str)
        .str.replace("US", "", regex=False)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
        .str.zfill(5)
        .str[:5]
    )

def normalize_county_name(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+county$", "", regex=True)
        .str.replace(r"\s+parish$", "", regex=True)
        .str.replace(r"\s+borough$", "", regex=True)
        .str.replace(r"\s+census area$", "", regex=True)
        .str.replace(r"\s+municipality$", "", regex=True)
        .str.replace(r"\.", "", regex=False)
        .str.strip()
    )

def normalize_state_name(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
    )

def title_case_location(series):
    return series.astype(str).str.strip().str.title()

In [53]:
# ---------------------------------------------------------
# COUNTY-LEVEL TECHNOECONOMIC TABLE
# suitability_scores_county.csv usually has:
# ['GHI', 'Protected_Land', 'Habitat', 'Slope',
#  'Population_Density', 'Distance_to_Substation',
#  'Land_Cover', 'County Name', 'State']
# ---------------------------------------------------------

tech = tech.copy()
social = social.copy()

tech.columns = [c.strip() for c in tech.columns]
social.columns = [c.strip() for c in social.columns]

# Social table GEOID
if "GEOID" not in social.columns:
    raise KeyError("Expected GEOID column in county_clean/social_factors_merged.csv")

social["GEOID"] = clean_geoid_5(social["GEOID"])

# Build normalized merge keys
tech["county_key"] = normalize_county_name(tech["County Name"])
tech["state_key"] = normalize_state_name(tech["State"])

social["county_key"] = normalize_county_name(social["County Name"])
social["state_key"] = normalize_state_name(social["State"])

# Attach GEOID from social table to tech table using county/state name
lookup = social[["GEOID", "county_key", "state_key"]].drop_duplicates()

tech = tech.merge(
    lookup,
    on=["county_key", "state_key"],
    how="left"
)

missing_geoid_rate = tech["GEOID"].isna().mean() * 100
print(f"Missing GEOID rate in tech after merge: {missing_geoid_rate:.2f}%")

if missing_geoid_rate > 0:
    print("\nSample unmatched rows:")
    display(tech.loc[tech["GEOID"].isna(), ["County Name", "State"]].drop_duplicates().head(10))

Missing GEOID rate in tech after merge: 4.10%

Sample unmatched rows:


,County Name,State
26,NaN,NaN


In [54]:
# Merge full county dataset
df = tech.merge(
    social,
    on="GEOID",
    how="inner",
    suffixes=("_tech", "_social")
)

print("Merged county dataset shape:", df.shape)
display(df.head())

Merged county dataset shape: (3275, 79)


,GHI,Protected_Land,Habitat,Slope,Population_Density,Distance_to_Substation,Land_Cover,County Name_tech,State_tech,county_key_tech,state_key_tech,GEOID,State_social,County Name_social,area km2,area mi2,FIPS State,FIPS County,Wind Capacity Intensity (MW / 1000 sq mile),Wind Project Intensity (Projects / 1000 sq mile),Wind Avg Capacity Intensity (MW / 1000 sq mile),GDP_2017,GDP_2018,GDP_2019,GDP_2020,GDP_2021,GDP_2022,Solar MW 1000 sq mile all,Solar Projects 1000 sq mile all,Solar MW Avg 1000 sq mile all,Solar MW 1000 sq mile small,Solar Projects 1000 sq mile small,Solar MW Avg 1000 sq mile small,Solar MW 1000 sq mile medium,Solar Projects 1000 sq mile medium,Solar MW Avg 1000 sq mile medium,Solar MW 1000 sq mile large,Solar Projects 1000 sq mile large,Solar MW Avg 1000 sq mile large,No. of Private Schools,Median Income,Total Unemployment,Unemployment Rate,Hispanic/Latino,White,Black/African American,American Indian/Alaska Native,Asian,Native Hawaiian/Other Pacific Islander,Others,Number of Existing Installs,Total Installed Capacity (kW),Median Installed Capacity (kW),Total Installed Capacity (kW/ 1000 sq mile),Median Installed Capacity (kW / sq mile),Number of Existing Installs / sq mile,democrat_percentage_vote,republican_percentage_vote,green_percentage_vote,libertarian_percentage_vote,other_percentage_vote,18-24 Less than high school graduate,18-24 High school graduate,18-24 Some college or associate's degree,18-24 Bachelor's degree or higher,25+ Less than 9th grade,"25+ 9th to 12th grade, no diploma",25+ High school graduate,"25+ Some college, no degree",25+ Associate's degree,25+ Bachelor's degree,25+ Graduate or professional degree,25+ High school graduate or higher,25+ Bachelor's degree or higher,Electric Commercial Rate,Electric Industrial Rate,Electric Residential Rate,county_key_social,state_key_social
0,15.0,91.547835,33.748285,91.029530,98.634118,53.241671,67.860969,Ballard,Kentucky,ballard,kentucky,21007,Kentucky,Ballard,708.542173,273.569550,21.0,7.0,NaN,NaN,NaN,27.28,25.76,27.77,28.16,32.05,31.53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52695,6410.0,5.5,0.012034,0.915373,0.026915,0.000906,0.003753,0.000000,0.039467,NaN,NaN,NaN,NaN,NaN,NaN,0.195266,0.794320,NaN,0.007574,0.002840,7.9,59.3,32.0,0.8,3.9,6.9,39.2,23.1,10.8,10.8,5.4,89.2,16.1,0.105367,0.063209,0.106003,ballard,kentucky
1,15.0,99.583685,82.840678,71.723980,96.682895,60.582990,85.691192,Bourbon,Kentucky,bourbon,kentucky,21017,Kentucky,Bourbon,755.280071,291.615146,21.0,17.0,NaN,NaN,NaN,46.76,49.41,47.59,43.80,43.69,44.24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,53277,16132.0,5.1,0.073375,0.828066,0.051007,0.001383,0.003802,0.000395,0.040984,NaN,NaN,NaN,NaN,NaN,NaN,0.341802,0.641916,NaN,0.010992,0.005289,17.8,46.1,32.2,3.9,6.6,7.6,34.3,22.5,5.9,12.6,10.5,85.8,23.1,0.105367,0.063209,0.106003,bourbon,kentucky
2,15.0,99.896136,39.236428,42.759798,98.239510,50.000000,61.839634,Butler,Kentucky,butler,kentucky,21031,Kentucky,Butler,1117.793121,431.582160,21.0,31.0,NaN,NaN,NaN,25.16,23.45,24.77,23.93,24.80,24.48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,48355,9894.0,5.5,0.047854,0.919085,0.001213,0.001778,0.001698,0.000243,0.026029,NaN,NaN,NaN,NaN,NaN,NaN,0.176250,0.810193,NaN,0.010291,0.003267,25.8,45.5,26.8,1.9,7.8,11.3,45.9,14.5,6.6,6.8,7.0,80.8,13.8,NaN,NaN,NaN,butler,kentucky
3,15.0,97.733883,29.826520,11.827322,93.352800,52.380053,56.737948,Estill,Kentucky,estill,kentucky,21065,Kentucky,Estill,662.202105,255.677557,21.0,65.0,NaN,NaN,NaN,15.24,15.21,15.59,15.08,14.86,15.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36336,11499.0,7.1,0.009249,0.961590,0.002118,0.001624,0.000494,0.000000,0.024077,NaN,NaN,NaN,NaN,NaN,NaN,0.207250,0.780055,NaN,0.009789,0.002906,29.2,31.8,38.0,1.0,12.6,13.1,42.9,14.2,6.6,7.3,3.3,74.3,10.6,0.105367,0.063209,0.106003,estill,kentucky
4,15.0,98.000821,49.732293,32.705892,99.466845,50.000000,71.555286,Fleming,Kentucky,fleming,kentucky,21069,Kentucky,Fleming,909.915069,351.320028,21.0,69.0,NaN,

In [55]:
# ---------------------------------------------------------
# EXACT COUNTY COLUMNS TO USE
# ---------------------------------------------------------

# -----------------------------
# Socio-political columns
# -----------------------------
SOCIAL_COLS = {
    "Black": "Black/African American",
    "Asian": "Asian",
    "Income": "Median Income",
    "Grad_Education": "25+ Graduate or professional degree",
    "GDP_per_Capita": None,   # computed below
}

# -----------------------------
# Techno-economic columns
# -----------------------------
TECH_COLS = {
    "GHI": "GHI",
    "Unprotected_Land": "Protected_Land",      # see note below
    "Slope": "Slope",
    "Population_Sparsity": "Population_Density"
}

print("Socio-political columns used:")
for k, v in SOCIAL_COLS.items():
    print(f"  {k} -> {v}")

print("\nTechno-economic columns used:")
for k, v in TECH_COLS.items():
    print(f"  {k} -> {v}")

Socio-political columns used:
  Black -> Black/African American
  Asian -> Asian
  Income -> Median Income
  Grad_Education -> 25+ Graduate or professional degree
  GDP_per_Capita -> None

Techno-economic columns used:
  GHI -> GHI
  Unprotected_Land -> Protected_Land
  Slope -> Slope
  Population_Sparsity -> Population_Density


In [56]:
# ---------------------------------------------------------
# VARIABLE CONSTRUCTION
# ---------------------------------------------------------

df = df.copy()

# GDP per capita
# Use latest GDP available in merged county social file
if "GDP_2022" in df.columns and "Median Income" in df.columns:
    pass

# We need county population proxy for GDP per capita if not already present.
# If a true GDP-per-capita column exists, use it.
gdp_pc_candidates = [
    "GDP per capita",
    "GDP_Per_Capita",
    "GDP_pc",
    "gdp_per_capita"
]

gdp_pc_found = next((c for c in gdp_pc_candidates if c in df.columns), None)

if gdp_pc_found is not None:
    df["GDP_per_Capita_used"] = pd.to_numeric(df[gdp_pc_found], errors="coerce")
else:
    # Fallback: if no explicit GDP-per-capita field exists, use GDP_2022 directly as proxy.
    # This is not ideal, but it keeps notebook runnable until Jenny clarifies.
    # If you later get a true GDP-per-capita column, replace this.
    if "GDP_2022" in df.columns:
        df["GDP_per_Capita_used"] = pd.to_numeric(df["GDP_2022"], errors="coerce")
        print("WARNING: No explicit GDP per capita column found; using GDP_2022 as temporary proxy.")
    else:
        raise KeyError("Could not find GDP per capita or GDP_2022 column.")

# Protected_Land in suitability score is a 'more buildable' style score in this dataset.
# Since your table says 'unprotected land', we will use the provided county suitability score column
# that corresponds to land buildability. Keep this documented clearly.
df["Unprotected_Land_used"] = pd.to_numeric(df["Protected_Land"], errors="coerce")

# Population sparsity:
# the table coefficient is for population sparsity, but the county suitability file has Population_Density.
# We will use that county-level suitability score column directly as the matching planning/siting score variable.
df["Population_Sparsity_used"] = pd.to_numeric(df["Population_Density"], errors="coerce")

# Other direct variables
df["GHI_used"] = pd.to_numeric(df["GHI"], errors="coerce")
df["Slope_used"] = pd.to_numeric(df["Slope"], errors="coerce")

df["Black_used"] = pd.to_numeric(df["Black/African American"], errors="coerce")
df["Asian_used"] = pd.to_numeric(df["Asian"], errors="coerce")
df["Income_used"] = pd.to_numeric(df["Median Income"], errors="coerce")
df["Grad_Education_used"] = pd.to_numeric(df["25+ Graduate or professional degree"], errors="coerce")

# Convert race shares to percent if still in decimals
for c in ["Black_used", "Asian_used"]:
    if df[c].dropna().max() <= 1.5:
        df[c] = df[c] * 100

display(df[[
    "GEOID",
    "County Name_social",
    "State_social",
    "GHI_used",
    "Unprotected_Land_used",
    "Slope_used",
    "Population_Sparsity_used",
    "Black_used",
    "Asian_used",
    "Income_used",
    "Grad_Education_used",
    "GDP_per_Capita_used"
]].head())

,GEOID,County Name_social,State_social,GHI_used,Unprotected_Land_used,Slope_used,Population_Sparsity_used,Black_used,Asian_used,Income_used,Grad_Education_used,GDP_per_Capita_used
0,21007,Ballard,Kentucky,15.0,91.547835,91.029530,98.634118,2.691511,0.375259,52695.0,5.4,31.53
1,21017,Bourbon,Kentucky,15.0,99.583685,71.723980,96.682895,5.100731,0.380209,53277.0,10.5,44.24
2,21031,Butler,Kentucky,15.0,99.896136,42.759798,98.239510,0.121251,0.169752,48355.0,7.0,24.48
3,21065,Estill,Kentucky,15.0,97.733883,11.827322,93.352800,0.211820,0.049425,36336.0,3.3,15.12
4,21069,Fleming,Kentucky,15.0,98.000821,32.705892,99.466845,0.981302,0.112717,48315.0,5.4,20.46


In [57]:
# ---------------------------------------------------------
# RAW PROPENSITY SCORES
# ---------------------------------------------------------

# Social-only version:
df["propensity_social_raw"] = (
    INTERCEPT
    + SOCIAL_COEFS["Black"] * df["Black_used"]
    + SOCIAL_COEFS["Asian"] * df["Asian_used"]
    + SOCIAL_COEFS["Income"] * df["Income_used"]
    + SOCIAL_COEFS["Grad_Education"] * df["Grad_Education_used"]
    + SOCIAL_COEFS["GDP_per_Capita"] * df["GDP_per_Capita_used"]
)

# Full version = techno-economic + socio-political
df["propensity_full_raw"] = (
    INTERCEPT
    + TECH_COEFS["GHI"] * df["GHI_used"]
    + TECH_COEFS["Unprotected_Land"] * df["Unprotected_Land_used"]
    + TECH_COEFS["Slope"] * df["Slope_used"]
    + TECH_COEFS["Population_Sparsity"] * df["Population_Sparsity_used"]
    + SOCIAL_COEFS["Black"] * df["Black_used"]
    + SOCIAL_COEFS["Asian"] * df["Asian_used"]
    + SOCIAL_COEFS["Income"] * df["Income_used"]
    + SOCIAL_COEFS["Grad_Education"] * df["Grad_Education_used"]
    + SOCIAL_COEFS["GDP_per_Capita"] * df["GDP_per_Capita_used"]
)

display(df[[
    "GEOID",
    "County Name_social",
    "State_social",
    "propensity_social_raw",
    "propensity_full_raw"
]].head())

,GEOID,County Name_social,State_social,propensity_social_raw,propensity_full_raw
0,21007,Ballard,Kentucky,24160.003247,24239.219266
1,21017,Bourbon,Kentucky,24437.026955,24513.813525
2,21031,Butler,Kentucky,22159.094356,22229.722008
3,21065,Estill,Kentucky,16636.648657,16698.731641
4,21069,Fleming,Kentucky,22138.998812,22207.044747


In [58]:
def minmax_scale(s):
    s = pd.to_numeric(s, errors="coerce")
    return (s - s.min()) / (s.max() - s.min())

df["social_score"] = minmax_scale(df["propensity_social_raw"])
df["full_score"] = minmax_scale(df["propensity_full_raw"])

display(df[[
    "GEOID",
    "County Name_social",
    "State_social",
    "propensity_social_raw",
    "social_score",
    "propensity_full_raw",
    "full_score"
]].head())

,GEOID,County Name_social,State_social,propensity_social_raw,social_score,propensity_full_raw,full_score
0,21007,Ballard,Kentucky,24160.003247,0.254149,24239.219266,0.254130
1,21017,Bourbon,Kentucky,24437.026955,0.258468,24513.813525,0.258413
2,21031,Butler,Kentucky,22159.094356,0.222950,22229.722008,0.222788
3,21065,Estill,Kentucky,16636.648657,0.136840,16698.731641,0.136521
4,21069,Fleming,Kentucky,22138.998812,0.222636,22207.044747,0.222434


In [59]:
# ---------------------------------------------------------
# FINAL CLEAN OUTPUTS
# Final CSV should only have:
#   county_name
#   score
# ---------------------------------------------------------

df["county_name"] = (
    title_case_location(df["County Name_social"]) + ", " + title_case_location(df["State_social"])
)

social_out = df[["county_name", "social_score"]].copy()
social_out = social_out.rename(columns={"social_score": "score"})
social_out = social_out.sort_values("county_name").reset_index(drop=True)

full_out = df[["county_name", "full_score"]].copy()
full_out = full_out.rename(columns={"full_score": "score"})
full_out = full_out.sort_values("county_name").reset_index(drop=True)

display(social_out.head())
display(full_out.head())

,county_name,score
0,"Abbeville, South Carolina",0.204355
1,"Acadia, Louisiana",0.180378
2,"Accomack, Virginia",0.239722
3,"Ada, Idaho",0.414953
4,"Adair, Iowa",0.292135


,county_name,score
0,"Abbeville, South Carolina",0.204317
1,"Acadia, Louisiana",0.180453
2,"Accomack, Virginia",0.239627
3,"Ada, Idaho",0.414832
4,"Adair, Iowa",0.292097


In [60]:
social_csv = "county_propensity_scores_social_only.csv"
full_csv = "county_propensity_scores_full.csv"

social_out.to_csv(social_csv, index=False)
full_out.to_csv(full_csv, index=False)

print("Wrote:", social_csv)
print("Wrote:", full_csv)

Wrote: county_propensity_scores_social_only.csv
Wrote: county_propensity_scores_full.csv


In [61]:
print("Social-only score summary:")
display(social_out["score"].describe())

print("\nFull score summary:")
display(full_out["score"].describe())

print("\nTop 10 counties by full score:")
display(full_out.sort_values("score", ascending=False).head(10))

print("\nTop 10 counties by social-only score:")
display(social_out.sort_values("score", ascending=False).head(10))

Social-only score summary:


count    3040.000000
mean        0.291609
std         0.108557
min         0.000000
25%         0.221126
50%         0.276643
75%         0.338375
max         1.000000
Name: score, dtype: float64


Full score summary:


count    3040.000000
mean        0.291535
std         0.108538
min         0.000000
25%         0.221083
50%         0.276731
75%         0.338341
max         1.000000
Name: score, dtype: float64


Top 10 counties by full score:


,county_name,score
1790,"Loudoun, Virginia",1.000000
2642,"Santa Clara, California",0.883521
2629,"San Mateo, California",0.858717
1861,"Marin, California",0.815350
1360,"Howard, Maryland",0.804808
87,"Arlington, Virginia",0.795551
834,"Douglas, Colorado",0.789257
2141,"Nassau, New York",0.783288
2621,"San Francisco, California",0.783252
1787,"Los Alamos, New Mexico",0.763683



Top 10 counties by social-only score:


,county_name,score
1790,"Loudoun, Virginia",1.000000
2642,"Santa Clara, California",0.883122
2629,"San Mateo, California",0.858879
1861,"Marin, California",0.815624
1360,"Howard, Maryland",0.805023
87,"Arlington, Virginia",0.795726
834,"Douglas, Colorado",0.789393
2141,"Nassau, New York",0.783516
2621,"San Francisco, California",0.783510
2106,"Morris, New Jersey",0.763563
